In [1]:
using NeuroDSL

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   cuSPARSE [b26da814-b3bc-49ef-b0ee-c816305aa060]
│   ChainRulesCoreExt [8ccf546c-b5ad-5491-b9b6-b62e2a4db961]
│   GPUArraysSparseArraysExt [555d850b-e4ac-5d9d-bfa0-17bb432b4efb]
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   NVML [611af6d1-644e-4c5d-bd58-854d7d1254b9]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   JLD2Ext [3dbd0623-ce14-52d8-8601-bc177a3b211d]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   ZygoteExt [3f48baa7-5843-56cc-8004-b8b725de387b]
│   FluxCUDAExt [dd41ee52-2073-581e-92e8-26baf003f19a]
│   cuSOLVER [887afef0-6a32-4de5-add4-7827692ba8fc]
│   CUDAExt [9e0aca61-9665-56cf-9642-d947ef6fc392]
│   StructArraysAdaptExt [f04e5bcb-ab32-5a64-8b64-c2cc4abec66e]
│   OneHotArrays [0b1bfda6-eb8a-41d2-88d8-f5af5cad476f]
│   CUDACore [bd0ed864-bdfe-4181-a5ed-ce625a5fdea2]
│   ZygoteDistancesExt [5865c103-18d1-586a-9b11-010bbc2260a8]
│   MLUtilsExt [447e3217-c189

In [2]:
using NeuroDSL

# Définir les opérateurs nécessaires
try
    @defop matmul (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        mul!(out, inputs[1], inputs[2])
    end
catch e
end
@defop identity out = inputs[1]

g = NeuroGraph()
activate!(g, :main)

@neuro g ns=:main operators=[:matmul, :relu, :identity] begin
    @node x = rand(10,5)
    @node w = rand(5,3)
    @node y = matmul(x, w)
    @node z = relu(y)
    @node loss = identity(z)
end

scan_summary(g, FULL_LLAMA_RULES)
plan = compile(g, CompilerConfig(rules=FULL_LLAMA_RULES, inspect=true))
println(plan)
result = plan(g, :loss)
println("Loss value: ", result)

✅ Op :matmul registered
✅ Op :identity registered
── scan_graph [:main] ──────────────────────────────────────────
  2 opportunités  |  8 nœuds  |  6 règles de calcul

  double_identity_elim               Δ=  1.0 ✓  nodes=[:identity_2, :loss]
  double_identity_elim               Δ=  1.0 ✓  nodes=[:z, :identity_2]

  Gain total estimé : Δ=2.0
────────────────────────────────────────────────────────
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 7
  Fusions     : 0  (7 → opérateurs fusionnés)
  Mémoire     : 7 slots  (pic réduit vs naïf)
  Recompiles  : 0 incrémentales
  Compilé il y a : 0.2s

Loss value: Float32[1.5708987 2.3135972 0.84510213; 1.6851925 2.122066 0.9088552; 0.96512336 2.3492951 1.0787303; 0.7402687 1.4411438 0.4224331; 1.7355173 1.7266387 1.0601325; 1.8158178 2.8676448 1.3761075; 1.0507176 1.7536631 0.5826569; 1.737232 2.8511777 1.2867138; 1.3598223 2.5791128

# tester sur deux spirales

# CPU 

In [3]:
using NeuroDSL, Random, Printf, LinearAlgebra

# ── Opérateurs ──────────────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end

# ── Règle de fusion ────────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(
    :add_relu_fusion,
    (:add, :relu),
    :fused_add_relu;
    cost_delta = 0.25f0,
    verified   = false
)

Random.seed!(42)
n_samples = 200
X = rand(Float32, n_samples, 2)
y = rand(Float32, n_samples, 1)

function build_mlp_cpu(name)
    g = NeuroGraph(device=Backend.CPUDevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, rand(Float32, 2, 10); namespace=name)
    set!(g, :b1, rand(Float32, 1, 10); namespace=name)
    set!(g, :W2, rand(Float32, 10, 1); namespace=name)
    set!(g, :b2, rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

compiler_rules = [ADD_RELU_FUSION_RULE]

# ── Modèles ─────────────────────────────────────────────────
g1 = build_mlp_cpu(:main)
plan = compile(g1, CompilerConfig(rules=compiler_rules))

g2 = build_mlp_cpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up (compile toutes les fonctions) ──────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark après warm-up ────────────────────────────────
println("=== Benchmark CPU (après warm-up) ===")
N = 100
t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.2f ms par appel (moyenne sur %d appels)\n", t1/N*1000, N)
@printf("Non compilé : %.2f ms par appel (moyenne sur %d appels)\n", t2/N*1000, N)
@printf("Accélération : x%.2f\n", (t2/N)/(t1/N))

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
=== Benchmark CPU (après warm-up) ===
Compilé    : 0.62 ms par appel (moyenne sur 100 appels)
Non compilé : 0.05 ms par appel (moyenne sur 100 appels)
Accélération : x0.08


# GPU

In [4]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs GPU ──────────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end

@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end

# ── Règle de fusion ────────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(
    :add_relu_fusion,
    (:add, :relu),
    :fused_add_relu;
    cost_delta = 0.25f0,
    verified   = false
)

# ── Données sur GPU ────────────────────────────────────────
Random.seed!(42)
n_samples = 200
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du modèle GPU ──────────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 10); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 10); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 10, 1); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

# ── Copie des paramètres entre deux graphes ─────────────────
function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Règles de compilation ──────────────────────────────────
compiler_rules = [ADD_RELU_FUSION_RULE]

# ── Modèle compilé ──────────────────────────────────────────
g1 = build_mlp_gpu(:main)
plan = compile(g1, CompilerConfig(rules=compiler_rules))

# ── Modèle non compilé (poids identiques) ───────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up (10 itérations) ──────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ───────────────────────────────────────────
println("=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

# ── Vérification des valeurs (CPU) ─────────────────────────
loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
=== Benchmark GPU (après warm-up) ===
Compilé    : 1.305 ms/appel (moyenne sur 50)
Non compilé : 0.909 ms/appel (moyenne sur 50)
Ralentissement : x1.43

✅ Pertes identiques sur GPU.


In [5]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs ──────────────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
# fused_matmul_relu est déjà défini dans dispatch.jl → on ne le redéfinit pas

# ── Règles de fusion ────────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(
    :add_relu_fusion,
    (:add, :relu),
    :fused_add_relu;
    cost_delta = 0.25f0,
    verified   = false
)

const MATMUL_RELU_FUSION_RULE = RewriteRule(
    :matmul_relu_fusion,
    (:matmul, :relu),
    :fused_matmul_relu;
    cost_delta = 0.30f0,
    verified   = false
)

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 200
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du modèle GPU ──────────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 10); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 10); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 10, 1); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

# ── Copie des paramètres ──────────────────────────────────
function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Règles de compilation (les deux fusions) ──────────────
compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_RELU_FUSION_RULE]

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé (poids identiques) ─────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  1 opportunités  |  12 nœuds  |  6 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]

  Gain total estimé : Δ=0.25
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 12
  Fusions     : 0  (12 → opérateurs fusionnés)
  Mémoire     : 12 slots  (pic réduit vs naïf)
  Recompiles  : 0 incrémentales
  Compilé il y a : 0.0s


=== Benchmark GPU (après warm-up) ===
Compilé    : 1.187 ms/appel (moyenne sur 50)
Non compilé : 0.988 ms/appel (moyenne sur 50)
Ralentissement : x1.20

✅ Pertes identiques sur GPU.


In [6]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs ──────────────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end

# ── Règles de fusion ────────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(
    :add_relu_fusion,
    (:add, :relu),
    :fused_add_relu;
    cost_delta = 0.25f0,
    verified   = false
)
const MATMUL_ADD_FUSION_RULE = RewriteRule(
    :matmul_add_fusion,
    (:matmul, :add),
    :fused_matmul_add;
    cost_delta = 0.20f0,
    verified   = false
)

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du modèle (3 couches) ─────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

# ── Copie des paramètres ──────────────────────────────────
function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Règles de compilation ──────────────────────────────────
compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE]

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé (poids identiques) ─────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic réduit vs naïf)
  Recompiles  : 0 incrémentales
  Compilé i

In [7]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

# ── Opérateurs fusionnés (uniquement ceux nécessaires) ────
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end

@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end

# ── Règles de fusion (seulement les deux sûres) ────────────
const ADD_RELU_FUSION_RULE = RewriteRule(
    :add_relu_fusion,
    (:add, :relu),
    :fused_add_relu;
    cost_delta = 0.25f0,
    verified   = false
)
const MATMUL_ADD_FUSION_RULE = RewriteRule(
    :matmul_add_fusion,
    (:matmul, :add),
    :fused_matmul_add;
    cost_delta = 0.20f0,
    verified   = false
)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE]
println("Règles actives : ", length(compiler_rules))

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du MLP 3 couches ──────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé ────────────────────────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
Règles actives : 2
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic réduit vs naïf)
  Recompiles  : 0 incrém

In [8]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

# ── Opérateurs fusionnés ──────────────────────────────────
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end

@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end

# ── Règles de fusion ───────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(
    :add_relu_fusion,
    (:add, :relu),
    :fused_add_relu;
    cost_delta = 0.25f0,
    verified   = false
)
const MATMUL_ADD_FUSION_RULE = RewriteRule(
    :matmul_add_fusion,
    (:matmul, :add),
    :fused_matmul_add;
    cost_delta = 0.20f0,
    verified   = false
)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE]

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du MLP 3 couches ──────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé (poids identiques) ─────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic réduit vs naïf)
  Recompiles  : 0 incrémentales
  Compilé i

# Raffinement

In [9]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

# ── Opérateurs fusionnés (existants + fused_matmul_relu) ──
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end
@defop fused_matmul_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= max.(temp, 0f0)
end

# ── Règles de fusion (nos 2 custom + matmul_relu par défaut) ──
const ADD_RELU_FUSION_RULE = RewriteRule(:add_relu_fusion, (:add, :relu), :fused_add_relu; cost_delta=0.25f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)
const MATMUL_RELU_FUSION_RULE = RewriteRule(:matmul_relu_fusion, (:matmul, :relu), :fused_matmul_relu; cost_delta=0.30f0)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE, MATMUL_RELU_FUSION_RULE]

# ── Données GPU (inchangées) ───────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du MLP 3 couches ──────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé ────────────────────────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
✅ Op :fused_matmul_relu registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic réduit vs naïf)
  Recomp

In [10]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

# ── Opérateurs fusionnés (sans fused_relu_matmul) ─────────
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end
@defop fused_matmul_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= max.(temp, 0f0)
end

# ── Règles de fusion (3 règles stables) ────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(:add_relu_fusion, (:add, :relu), :fused_add_relu; cost_delta=0.25f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)
const MATMUL_RELU_FUSION_RULE = RewriteRule(:matmul_relu_fusion, (:matmul, :relu), :fused_matmul_relu; cost_delta=0.30f0)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE, MATMUL_RELU_FUSION_RULE]

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du MLP 3 couches ──────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé ────────────────────────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
✅ Op :fused_matmul_relu registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic réduit vs naïf)
  Recomp

# sécurités des règles

In [11]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

# ── Opérateurs fusionnés ──────────────────────────────────
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end
@defop fused_matmul_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= max.(temp, 0f0)
end
@defop fused_relu_matmul (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(A)
    temp .= max.(A, 0f0)
    LinearAlgebra.mul!(out, temp, tb ? B' : B)
end
@defop fused_linear_gelu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= temp .* (1f0 ./ (1f0 .+ exp.(-1.702f0 .* temp)))
end
@defop fused_matmul_sigmoid (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= 1f0 ./ (1f0 .+ exp.(-temp))
end
@defop scaled_matmul (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    factor = get(attrs, :factor, 1.0f0)
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .*= factor
end

# ── Règles avec conditions de sécurité ──────────────────────
# Condition générique : vérifie que le deuxième argument d'un matmul est une matrice 2D (pas un biais)
_is_weight_matrix(g, chain, ns) = begin
    length(chain) >= 1 || return false
    rule = get(g.rules[ns], chain[end], nothing)
    rule === nothing && return false
    length(rule.inputs) >= 2 || return false
    inp2 = g.nodes[ns][rule.inputs[2]].value
    ndims(inp2) == 2 && size(inp2, 2) > 1
end

# add_relu : toujours sûr
const ADD_RELU_FUSION_RULE = RewriteRule(:add_relu_fusion, (:add, :relu), :fused_add_relu; cost_delta=0.25f0, verified=false)

# matmul_add : on vérifie que le troisième input (bias) existe et a la bonne dimension (1×n)
function _safe_matmul_add(g, chain, ns)
    length(chain) >= 1 || return false
    rule = get(g.rules[ns], chain[end], nothing)
    rule === nothing && return false
    length(rule.inputs) >= 2 || return false
    # Vérifie que le biais (inputs[3] si présent) est 2D avec une dimension 1 (broadcastable)
    if length(rule.inputs) >= 3
        bias = g.nodes[ns][rule.inputs[3]].value
        ndims(bias) == 2 && (size(bias,1) == 1 || size(bias,2) == 1) || return false
    end
    true
end
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0, condition=_safe_matmul_add)

# matmul_relu : sûr car pas de biais impliqué
const MATMUL_RELU_FUSION_RULE = RewriteRule(:matmul_relu_fusion, (:matmul, :relu), :fused_matmul_relu; cost_delta=0.30f0, verified=false)

# relu_matmul : on ajoute une condition pour éviter les fusions avec un biais
function _safe_relu_matmul(g, chain, ns)
    length(chain) == 2 || return false  # chaîne [relu, matmul]
    rule_matmul = get(g.rules[ns], chain[2], nothing)
    rule_matmul === nothing && return false
    length(rule_matmul.inputs) >= 2 || return false
    # Le deuxième input du matmul doit être une matrice de poids (2D, taille >1)
    inp2 = g.nodes[ns][rule_matmul.inputs[2]].value
    ndims(inp2) == 2 && size(inp2,2) > 1 || return false
    # Vérifier que le matmul n'est pas suivi d'un add (c.-à-d. que son unique consommateur n'est pas un add)
    cons = consumers(g, chain[2]; ns=ns)
    if length(cons) == 1
        op_cons = op_of(g, cons[1]; ns=ns)
        op_cons == :add && return false
    end
    true
end
const RELU_MATMUL_FUSION_RULE = RewriteRule(:relu_matmul_fusion, (:relu, :matmul), :fused_relu_matmul; cost_delta=0.20f0, condition=_safe_relu_matmul)

# Règles sans risque (conditions nil)
const LINEAR_GELU_FUSION_RULE = RewriteRule(:linear_gelu_fusion, (:matmul, :gelu), :fused_linear_gelu; cost_delta=0.35f0, verified=false)
const MATMUL_SIGMOID_FUSION_RULE = RewriteRule(:matmul_sigmoid_fusion, (:matmul, :sigmoid), :fused_matmul_sigmoid; cost_delta=0.25f0, verified=false)
const SCALE_ABSORB_MATMUL_RULE = RewriteRule(:scale_absorb_matmul, (:scale, :matmul), :scaled_matmul; cost_delta=0.15f0, verified=false)
const MATMUL_SCALE_ABSORB_RULE = RewriteRule(:matmul_scale_absorb, (:matmul, :scale), :scaled_matmul; cost_delta=0.15f0, verified=false)

# Compilation de toutes les règles (celles qui peuvent s'appliquer sans danger)
compiler_rules = [
    ADD_RELU_FUSION_RULE,
    MATMUL_ADD_FUSION_RULE,
    MATMUL_RELU_FUSION_RULE,
    RELU_MATMUL_FUSION_RULE,
    LINEAR_GELU_FUSION_RULE,
    MATMUL_SIGMOID_FUSION_RULE,
    SCALE_ABSORB_MATMUL_RULE,
    MATMUL_SCALE_ABSORB_RULE
]

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── Construction du MLP 3 couches ──────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé ────────────────────────────────────
g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark GPU ─────────────────────────────────────────
println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU avec $(length(compiler_rules)) règles sécurisées.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
✅ Op :fused_matmul_relu registered
✅ Op :fused_relu_matmul registered
✅ Op :fused_linear_gelu registered
✅ Op :fused_matmul_sigmoid registered
✅ Op :scaled_matmul registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Na


=== Benchmark GPU (après warm-up) ===
Compilé    : 1.427 ms/appel (moyenne sur 50)
Non compilé : 1.253 ms/appel (moyenne sur 50)
Ralentissement : x1.14

✅ Pertes identiques sur GPU avec 8 règles sécurisées.


In [12]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs nécessaires ─────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end
@defop fused_matmul_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= max.(temp, 0f0)
end

# ── Règles actives ─────────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(:add_relu_fusion, (:add, :relu), :fused_add_relu; cost_delta=0.25f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)
const MATMUL_RELU_FUSION_RULE = RewriteRule(:matmul_relu_fusion, (:matmul, :relu), :fused_matmul_relu; cost_delta=0.30f0)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE, MATMUL_RELU_FUSION_RULE]

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 500
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── MLP 3 couches ──────────────────────────────────────────
function build_mlp_gpu(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 20); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 20, 20); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 20); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 20, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Compilation et benchmark ────────────────────────────────
g1 = build_mlp_gpu(:main)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

g2 = build_mlp_gpu(:raw)
copy_params!(g2, g1, :raw, :main)

for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

println("\n=== Benchmark GPU (après warm-up) ===")
N = 50
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:main)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss1_cpu = Array(plan(g1, :loss))
loss2_cpu = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss1_cpu, loss2_cpu; atol=1e-3)
println("\n✅ Pertes identiques sur GPU.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
✅ Op :fused_matmul_relu registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic réduit vs naïf)
  Recomp


Compilé    : 1.195 ms/appel (moyenne sur 50)
Non compilé : 1.221 ms/appel (moyenne sur 50)
Accélération : x1.02

✅ Pertes identiques sur GPU.


# 2 Spirales

In [13]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs nécessaires ─────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end
@defop fused_matmul_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= max.(temp, 0f0)
end

# ── Règles actives ─────────────────────────────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(:add_relu_fusion, (:add, :relu), :fused_add_relu; cost_delta=0.25f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)
const MATMUL_RELU_FUSION_RULE = RewriteRule(:matmul_relu_fusion, (:matmul, :relu), :fused_matmul_relu; cost_delta=0.30f0)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE, MATMUL_RELU_FUSION_RULE]

# ── Données Two-Spirals (simulées ici, à remplacer par ton dataset) ──
Random.seed!(42)
n_samples = 1000
X = CUDA.rand(Float32, n_samples, 2)          # entrées 2D
y = CUDA.rand(Float32, n_samples, 1)          # sortie binaire (0 ou 1)

# ── Construction du MLP (2 couches, plus gros) ──────────────
function build_two_spirals_mlp(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 64); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 64); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 64, 32); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 32); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 32, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g_comp = build_two_spirals_mlp(:compiled)
println("=== Opportunités de fusion (Two-Spirals) ===")
scan_summary(g_comp, compiler_rules)
println("\n=== Compilation ===")
plan = compile(g_comp, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé (poids identiques) ─────────────────
g_raw = build_two_spirals_mlp(:raw)
copy_params!(g_raw, g_comp, :raw, :compiled)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g_comp, :loss)
    demand!(g_raw, :loss; ctx_store=CtxStore())
end

# ── Benchmark sur plusieurs itérations ──────────────────────
println("\n=== Benchmark Two-Spirals (10 itérations) ===")
N = 10
CUDA.@sync t_comp = @elapsed for _ in 1:N
    set!(g_comp, :x, X; namespace=:compiled)
    plan(g_comp, :loss)
end
CUDA.@sync t_raw = @elapsed for _ in 1:N
    set!(g_raw, :x, X; namespace=:raw)
    demand!(g_raw, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t_comp/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t_raw/N*1000, N)
@printf("Accélération : x%.2f\n", (t_raw/N) / (t_comp/N))

loss_comp = Array(plan(g_comp, :loss))
loss_raw  = Array(demand!(g_raw, :loss; ctx_store=CtxStore()))
@assert isapprox(loss_comp, loss_raw; atol=1e-3)
println("✅ Pertes identiques.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
✅ Op :fused_matmul_relu registered
=== Opportunités de fusion (Two-Spirals) ===
── scan_graph [:compiled] ──────────────────────────────────────────
  5 opportunités  |  17 nœuds  |  9 règles de calcul

  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=1.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :compiled
  Nœuds       : 14
  Fusions     : 3  (14 → opérateurs fusionnés)
  Mémoire     : 14 slots  (pic ré

In [14]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_matmul_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    temp .+= bias
    out .= max.(temp, 0f0)
end

# ── Règle triple uniquement ────────────────────────────────
const MATMUL_ADD_RELU_FUSION_RULE = RewriteRule(:matmul_add_relu_fusion, (:matmul, :add, :relu), :fused_matmul_add_relu; cost_delta=0.50f0)

compiler_rules = [MATMUL_ADD_RELU_FUSION_RULE]

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 1000
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── MLP 3 couches ──────────────────────────────────────────
function build_mlp(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 64); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 64); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 64, 32); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 32); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 32, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp(:compiled)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé ────────────────────────────────────
g2 = build_mlp(:raw)
copy_params!(g2, g1, :raw, :compiled)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark ─────────────────────────────────────────────
println("\n=== Benchmark (10 itérations) ===")
N = 10
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:compiled)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss_comp = Array(plan(g1, :loss))
loss_raw  = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss_comp, loss_raw; atol=1e-3)
println("\n✅ Pertes identiques.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_matmul_add_relu registered
=== Opportunités de fusion ===
── scan_graph [:compiled] ──────────────────────────────────────────
  2 opportunités  |  17 nœuds  |  9 règles de calcul

  matmul_add_relu_fusion             Δ=  0.5    nodes=[:h1, :h2, :h3]
  matmul_add_relu_fusion             Δ=  0.5    nodes=[:h4, :h5, :h6]

  Gain total estimé : Δ=1.0
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :compiled
  Nœuds       : 13
  Fusions     : 2  (13 → opérateurs fusionnés)
  Mémoire     : 13 slots  (pic réduit vs naïf)
  Recompiles  : 0 incrémentales
  Compilé il y a : 0.0s


=== Benchmark (10 itérations) ===
Compilé    : 1.552 ms/appel (moyenne sur 10)
Non compilé : 1.553 ms/appel (moyenne sur 10)
Accélération : x1.00

✅ Pertes identiques.


In [15]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# ── Opérateurs de base ─────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end

# ── Opérateurs fusionnés (tous) ────────────────────────────
@defop fused_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    @. out = max(inputs[1] + inputs[2], 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end
@defop fused_matmul_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    out .= max.(temp, 0f0)
end
@defop fused_matmul_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    temp .+= bias
    out .= max.(temp, 0f0)
end

# ── Règles de fusion (doubles + triples) ──────────────────
const ADD_RELU_FUSION_RULE = RewriteRule(:add_relu_fusion, (:add, :relu), :fused_add_relu; cost_delta=0.25f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)
const MATMUL_RELU_FUSION_RULE = RewriteRule(:matmul_relu_fusion, (:matmul, :relu), :fused_matmul_relu; cost_delta=0.30f0)
const MATMUL_ADD_RELU_FUSION_RULE = RewriteRule(:matmul_add_relu_fusion, (:matmul, :add, :relu), :fused_matmul_add_relu; cost_delta=0.50f0)

compiler_rules = [ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE, MATMUL_RELU_FUSION_RULE, MATMUL_ADD_RELU_FUSION_RULE]

# ── Données GPU ────────────────────────────────────────────
Random.seed!(42)
n_samples = 1000
X = CUDA.rand(Float32, n_samples, 2)
y = CUDA.rand(Float32, n_samples, 1)

# ── MLP 3 couches ──────────────────────────────────────────
function build_mlp(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, CUDA.rand(Float32, 2, 64); namespace=name)
    set!(g, :b1, CUDA.rand(Float32, 1, 64); namespace=name)
    set!(g, :W2, CUDA.rand(Float32, 64, 32); namespace=name)
    set!(g, :b2, CUDA.rand(Float32, 1, 32); namespace=name)
    set!(g, :W3, CUDA.rand(Float32, 32, 1); namespace=name)
    set!(g, :b3, CUDA.rand(Float32, 1, 1); namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

function copy_params!(g_dest, g_src, dest_ns, src_ns)
    for sym in [:W1, :b1, :W2, :b2, :W3, :b3]
        val = node(g_src, sym; namespace=src_ns).value
        set!(g_dest, sym, copy(val); namespace=dest_ns)
    end
end

# ── Modèle compilé ────────────────────────────────────────
g1 = build_mlp(:compiled)
println("=== Opportunités de fusion ===")
scan_summary(g1, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g1, CompilerConfig(rules=compiler_rules))
println(plan)

# ── Modèle non compilé ────────────────────────────────────
g2 = build_mlp(:raw)
copy_params!(g2, g1, :raw, :compiled)

# ── Warm-up ───────────────────────────────────────────────
for _ in 1:10
    plan(g1, :loss)
    demand!(g2, :loss; ctx_store=CtxStore())
end

# ── Benchmark ─────────────────────────────────────────────
println("\n=== Benchmark (10 itérations) ===")
N = 10
CUDA.@sync t1 = @elapsed for _ in 1:N
    set!(g1, :x, X; namespace=:compiled)
    plan(g1, :loss)
end
CUDA.@sync t2 = @elapsed for _ in 1:N
    set!(g2, :x, X; namespace=:raw)
    demand!(g2, :loss; ctx_store=CtxStore())
end

@printf("Compilé    : %.3f ms/appel (moyenne sur %d)\n", t1/N*1000, N)
@printf("Non compilé : %.3f ms/appel (moyenne sur %d)\n", t2/N*1000, N)
if t1 < t2
    @printf("Accélération : x%.2f\n", (t2/N)/(t1/N))
else
    @printf("Ralentissement : x%.2f\n", (t1/N)/(t2/N))
end

loss_comp = Array(plan(g1, :loss))
loss_raw  = Array(demand!(g2, :loss; ctx_store=CtxStore()))
@assert isapprox(loss_comp, loss_raw; atol=1e-3)
println("\n✅ Pertes identiques.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_add_relu registered
✅ Op :fused_matmul_add registered
✅ Op :fused_matmul_relu registered
✅ Op :fused_matmul_add_relu registered
=== Opportunités de fusion ===
── scan_graph [:compiled] ──────────────────────────────────────────
  7 opportunités  |  17 nœuds  |  9 règles de calcul

  matmul_add_relu_fusion             Δ=  0.5    nodes=[:h1, :h2, :h3]
  matmul_add_relu_fusion             Δ=  0.5    nodes=[:h4, :h5, :h6]
  add_relu_fusion                    Δ= 0.25    nodes=[:h2, :h3]
  add_relu_fusion                    Δ= 0.25    nodes=[:h5, :h6]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h1, :h2]
  matmul_add_fusion                  Δ=  0.2    nodes=[:h4, :h5]
  matmul_add_fusion                  Δ=  0.2    nodes=[:out, :out_b]

  Gain total estimé : Δ=2.1
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚

In [16]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA, Flux

n_in, n_h1, n_h2, n_out = 2, 64, 32, 1
n_samples = 1000

# ── Données communes ────────────────────────────────────
Random.seed!(42)
X = CUDA.rand(Float32, n_samples, n_in)
y = CUDA.rand(Float32, n_samples, n_out)

# ── Poids partagés (format NeuroDSL : entrée × sortie) ──
Random.seed!(123)
W1 = CUDA.rand(Float32, n_in, n_h1)
b1 = CUDA.rand(Float32, 1, n_h1)
W2 = CUDA.rand(Float32, n_h1, n_h2)
b2 = CUDA.rand(Float32, 1, n_h2)
W3 = CUDA.rand(Float32, n_h2, n_out)
b3 = CUDA.rand(Float32, 1, n_out)

# ═══════════════════════════════════════════════════════════
# NeuroDSL compilé
# ═══════════════════════════════════════════════════════════
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_matmul_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, tb ? B' : B)
    temp .+= bias
    out .= max.(temp, 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    tb = get(attrs, :trans_b, false)
    LinearAlgebra.mul!(out, A, tb ? B' : B)
    out .+= bias
end

const MATMUL_ADD_RELU_FUSION_RULE = RewriteRule(:matmul_add_relu_fusion, (:matmul, :add, :relu), :fused_matmul_add_relu; cost_delta=0.50f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)

compiler_rules = [MATMUL_ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE]

function build_neurodsl(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, W1; namespace=name)
    set!(g, :b1, b1; namespace=name)
    set!(g, :W2, W2; namespace=name)
    set!(g, :b2, b2; namespace=name)
    set!(g, :W3, W3; namespace=name)
    set!(g, :b3, b3; namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    addrule!(g, GraphRule(:loss, [:out_b, :target], :mse_loss; namespace=name))
    return g
end

g_neuro = build_neurodsl(:main)
plan = compile(g_neuro, CompilerConfig(rules=compiler_rules))

# ═══════════════════════════════════════════════════════════
# Flux.jl (poids et données sur GPU)
# ═══════════════════════════════════════════════════════════
W1_f = CuArray(Array(W1)')   # (64, 2)
W2_f = CuArray(Array(W2)')   # (32, 64)
W3_f = CuArray(Array(W3)')   # (1, 32)

m_flux = Chain(
    Dense(W1_f, CuArray(vec(Array(b1))), relu),
    Dense(W2_f, CuArray(vec(Array(b2))), relu),
    Dense(W3_f, CuArray(vec(Array(b3)))),
)

X_flux = X'        # (2, 1000)
y_flux = y'        # (1, 1000)

loss_flux(ŷ, y) = Flux.mse(ŷ, y)

# ═══════════════════════════════════════════════════════════
# Benchmark
# ═══════════════════════════════════════════════════════════
println("=== Comparaison NeuroDSL (compilé) vs Flux.jl ===")

for _ in 1:10
    plan(g_neuro, :loss)
    loss_flux(m_flux(X_flux), y_flux)
end

N = 50
CUDA.@sync t_neuro = @elapsed for _ in 1:N
    set!(g_neuro, :x, X; namespace=:main)
    plan(g_neuro, :loss)
end
CUDA.@sync t_flux = @elapsed for _ in 1:N
    loss_flux(m_flux(X_flux), y_flux)
end

t_neuro_avg = t_neuro / N * 1000
t_flux_avg = t_flux / N * 1000

@printf("NeuroDSL (compilé) : %.4f ms/appel\n", t_neuro_avg)
@printf("Flux.jl            : %.4f ms/appel\n", t_flux_avg)

if t_neuro < t_flux
    @printf("✅ NeuroDSL est x%.2f plus rapide que Flux.jl\n", (t_flux/N) / (t_neuro/N))
else
    @printf("⚠️ Flux.jl est x%.2f plus rapide que NeuroDSL\n", (t_neuro/N) / (t_flux/N))
end

loss_neuro = Array(plan(g_neuro, :loss))[]
loss_flux_val = loss_flux(m_flux(X_flux), y_flux)[]
@assert isapprox(loss_neuro, loss_flux_val, atol=1e-3)
println("✅ Pertes identiques.")

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_matmul_add_relu registered
✅ Op :fused_matmul_add registered
=== Comparaison NeuroDSL (compilé) vs Flux.jl ===
NeuroDSL (compilé) : 0.9057 ms/appel
Flux.jl            : 1.1246 ms/appel
✅ NeuroDSL est x1.24 plus rapide que Flux.jl
✅ Pertes identiques.


In [17]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

n_in, n_h1, n_h2, n_out = 2, 64, 32, 1
n_samples = 1000

Random.seed!(42)
X = CUDA.rand(Float32, n_samples, n_in)
y = CUDA.rand(Float32, n_samples, n_out)

Random.seed!(123)
W1 = CUDA.rand(Float32, n_in, n_h1)
b1 = CUDA.rand(Float32, 1, n_h1)
W2 = CUDA.rand(Float32, n_h1, n_h2)
b2 = CUDA.rand(Float32, 1, n_h2)
W3 = CUDA.rand(Float32, n_h2, n_out)
b3 = CUDA.rand(Float32, 1, n_out)

# ── NeuroDSL compilé ──────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_matmul_add_relu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    temp = similar(out)
    LinearAlgebra.mul!(temp, A, B)
    temp .+= bias
    out .= max.(temp, 0f0)
end
@defop fused_matmul_add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B, bias = inputs[1], inputs[2], inputs[3]
    LinearAlgebra.mul!(out, A, B)
    out .+= bias
end

const MATMUL_ADD_RELU_FUSION_RULE = RewriteRule(:matmul_add_relu_fusion, (:matmul, :add, :relu), :fused_matmul_add_relu; cost_delta=0.50f0)
const MATMUL_ADD_FUSION_RULE = RewriteRule(:matmul_add_fusion, (:matmul, :add), :fused_matmul_add; cost_delta=0.20f0)
compiler_rules = [MATMUL_ADD_RELU_FUSION_RULE, MATMUL_ADD_FUSION_RULE]

function build_neurodsl(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, X; namespace=name)
    set!(g, :target, y; namespace=name)
    set!(g, :W1, W1; namespace=name)
    set!(g, :b1, b1; namespace=name)
    set!(g, :W2, W2; namespace=name)
    set!(g, :b2, b2; namespace=name)
    set!(g, :W3, W3; namespace=name)
    set!(g, :b3, b3; namespace=name)

    addrule!(g, GraphRule(:h1,   [:x, :W1],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h2,   [:h1, :b1],  :add;      namespace=name))
    addrule!(g, GraphRule(:h3,   [:h2],        :relu;     namespace=name))
    addrule!(g, GraphRule(:h4,   [:h3, :W2],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:h5,   [:h4, :b2],  :add;      namespace=name))
    addrule!(g, GraphRule(:h6,   [:h5],        :relu;     namespace=name))
    addrule!(g, GraphRule(:out,  [:h6, :W3],   :matmul;   namespace=name))
    addrule!(g, GraphRule(:out_b,[:out, :b3],  :add;      namespace=name))
    return g
end

g_neuro = build_neurodsl(:main)
plan = compile(g_neuro, CompilerConfig(rules=compiler_rules))

# ── Flux.jl (tout GPU) ────────────────────────────────────
W1_f = W1'
b1_f = vec(b1)
W2_f = W2'
b2_f = vec(b2)
W3_f = W3'
b3_f = vec(b3)

function flux_forward(x)
    x = W1_f * x .+ b1_f
    x = max.(x, 0f0)
    x = W2_f * x .+ b2_f
    x = max.(x, 0f0)
    x = W3_f * x .+ b3_f
    return x
end

X_flux = X'

# ── Warm-up avec invalidation ─────────────────────────────
for _ in 1:100
    set!(g_neuro, :x, X)
    plan(g_neuro, :out_b)
    flux_forward(X_flux)
end

# ── Benchmark ─────────────────────────────────────────────
N = 200
CUDA.@sync t_neuro = @elapsed for _ in 1:N
    set!(g_neuro, :x, X)
    plan(g_neuro, :out_b)
end
CUDA.@sync t_flux = @elapsed for _ in 1:N
    flux_forward(X_flux)
end

t_neuro_avg = t_neuro / N * 1000
t_flux_avg = t_flux / N * 1000

@printf("NeuroDSL (compilé) : %.4f ms/appel\n", t_neuro_avg)
@printf("Flux.jl            : %.4f ms/appel\n", t_flux_avg)

if t_neuro < t_flux
    @printf("✅ NeuroDSL est x%.2f plus rapide que Flux\n", (t_flux/N) / (t_neuro/N))
else
    @printf("⚠️ Flux est x%.2f plus rapide que NeuroDSL\n", (t_neuro/N) / (t_flux/N))
end

# ── Vérification avec diagnostic ──────────────────────────
set!(g_neuro, :x, X)
out_neuro = Array(plan(g_neuro, :out_b))
out_flux  = Array(flux_forward(X_flux))'

println("Taille out_neuro : ", size(out_neuro))
println("Taille out_flux  : ", size(out_flux))

diff = maximum(abs.(out_neuro .- out_flux))
println("Différence maximale : ", diff)

if diff < 1e-3
    println("✅ Sorties identiques (tolérance 1e-3)")
else
    println("⚠️ Écart supérieur à 1e-3 — vérifier les calculs")
end

✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_matmul_add_relu registered
✅ Op :fused_matmul_add registered
NeuroDSL (compilé) : 0.3665 ms/appel
Flux.jl            : 0.6096 ms/appel
✅ NeuroDSL est x1.66 plus rapide que Flux
Taille out_neuro : (1000, 1)
Taille out_flux  : (1000, 1)
Différence maximale : 0.00012207031
✅ Sorties identiques (tolérance 1e-3)


# Transformer

In [18]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

# Enregistrer flash_attn si nécessaire
if !haskey(NeuroDSL.CUSTOM_OPS, :flash_attn)
    NeuroDSL.register_op!(:flash_attn, NeuroDSL._dispatch_flash_attn!)
end

# Conditions (toujours vraies pour le test)
_is_qkv_pattern(g, chain, ns) = true
_is_swiglu_pattern(g, chain, ns) = true
_is_sdpa_pattern(g, chain, ns) = true

# Dimensions
n_heads, d_model, d_head, d_ff = 8, 512, 64, 2048
seq_len = 128

# Données d’entrée
Random.seed!(42)
x = CUDA.rand(Float32, seq_len, d_model)

# Poids (Q, K, V séparés pour que la fusion QKV s’applique)
W_q = CUDA.rand(Float32, d_model, d_model)      # (512, 512)
W_k = CUDA.rand(Float32, d_model, d_model)
W_v = CUDA.rand(Float32, d_model, d_model)
W_o = CUDA.rand(Float32, d_model, d_model)      # (512, 512)

# Poids SwiGLU
W_up   = CUDA.rand(Float32, d_model, d_ff)      # (512, 2048)
W_gate = CUDA.rand(Float32, d_model, d_ff)
W_down = CUDA.rand(Float32, d_ff, d_model)      # (2048, 512)

# ── Opérateurs fusionnés ────────────────────────────────────
@defop identity (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
end
@defop add (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    out .= inputs[1] .+ inputs[2]
end
@defop fused_qkv_projection (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]   # x, W_qkv
    LinearAlgebra.mul!(out, A, B)
end
@defop fused_swiglu (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    gate, up = inputs[1], inputs[2]
    swiglu_fwd!(dev, out, gate, up)
end
@defop fused_sdpa (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    Q, K, V = inputs[1], inputs[2], inputs[3]
    d_head = get(attrs, :d_head, size(Q,2))
    causal = get(attrs, :causal, true)
    flash_attn_fwd!(dev, out, Q, K, V, d_head; causal=causal)
end

# ── Règles de fusion ───────────────────────────────────────
const QKV_FUSION_RULE = RewriteRule(:qkv_projection_fusion, (:matmul,), :fused_qkv_projection; cost_delta=0.50f0, condition=_is_qkv_pattern)
const SWIGLU_FUSION_RULE = RewriteRule(:swiglu_fusion, (:matmul, :silu), :fused_swiglu; cost_delta=0.55f0, condition=_is_swiglu_pattern)
const SDPA_FUSION_RULE = RewriteRule(:sdpa_fusion, (:matmul, :scale, :softmax, :matmul), :fused_sdpa; cost_delta=0.70f0, condition=_is_sdpa_pattern)

compiler_rules = vcat(NeuroDSL.DEFAULT_RULES, [QKV_FUSION_RULE, SWIGLU_FUSION_RULE, SDPA_FUSION_RULE])

# ── Construction du bloc ───────────────────────────────────
function build_llama_block(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)

    # Nœuds d’entrée et poids
    set!(g, :x, x; namespace=name)
    set!(g, :W_q, W_q; namespace=name)
    set!(g, :W_k, W_k; namespace=name)
    set!(g, :W_v, W_v; namespace=name)
    set!(g, :W_o, W_o; namespace=name)
    set!(g, :W_up, W_up; namespace=name)
    set!(g, :W_gate, W_gate; namespace=name)
    set!(g, :W_down, W_down; namespace=name)

    # Projections Q, K, V (trois matmuls distincts → fusionnables en QKV)
    addrule!(g, GraphRule(:Q, [:x, :W_q], :matmul; namespace=name))
    addrule!(g, GraphRule(:K, [:x, :W_k], :matmul; namespace=name))
    addrule!(g, GraphRule(:V, [:x, :W_v], :matmul; namespace=name))

    # Attention (flash_attn)
    addrule!(g, GraphRule(:attn_out, [:Q, :K, :V], :flash_attn;
                         attrs=Dict(:d_head=>d_model, :causal=>true), namespace=name))

    # Projection de sortie
    addrule!(g, GraphRule(:proj_out, [:attn_out, :W_o], :matmul; namespace=name))

    # Résidu 1
    addrule!(g, GraphRule(:residual, [:proj_out, :x], :add; namespace=name))

    # SwiGLU
    addrule!(g, GraphRule(:gate, [:x, :W_gate], :matmul; namespace=name))
    addrule!(g, GraphRule(:up,   [:x, :W_up],   :matmul; namespace=name))
    addrule!(g, GraphRule(:ffn,  [:gate, :up],  :swiglu; namespace=name))
    addrule!(g, GraphRule(:down, [:ffn, :W_down], :matmul; namespace=name))

    # Résidu 2 (sortie finale)
    addrule!(g, GraphRule(:ffn_out, [:down, :x], :add; namespace=name))

    return g
end

# ── Compilation et benchmark ─────────────────────────────────
g_neuro = build_llama_block(:main)
println("=== Opportunités de fusion ===")
scan_summary(g_neuro, compiler_rules)

println("\n=== Compilation ===")
plan = compile(g_neuro, CompilerConfig(rules=compiler_rules))
println(plan)

# Warm-up
for _ in 1:10
    set!(g_neuro, :x, x)
    plan(g_neuro, :ffn_out)
end

# Mesure
N = 50
CUDA.@sync t_neuro = @elapsed for _ in 1:N
    set!(g_neuro, :x, x)
    plan(g_neuro, :ffn_out)
end
t_neuro_avg = t_neuro / N * 1000
@printf("NeuroDSL (compilé) : %.4f ms/appel\n", t_neuro_avg)

✅ Op :flash_attn registered
✅ Op :identity registered
✅ Op :add registered
✅ Op :fused_qkv_projection registered
✅ Op :fused_swiglu registered
✅ Op :fused_sdpa registered
=== Opportunités de fusion ===
── scan_graph [:main] ──────────────────────────────────────────
  7 opportunités  |  19 nœuds  |  11 règles de calcul

  qkv_projection_fusion              Δ=  0.5    nodes=[:proj_out]
  qkv_projection_fusion              Δ=  0.5    nodes=[:V]
  qkv_projection_fusion              Δ=  0.5    nodes=[:gate]
  qkv_projection_fusion              Δ=  0.5    nodes=[:K]
  qkv_projection_fusion              Δ=  0.5    nodes=[:down]
  qkv_projection_fusion              Δ=  0.5    nodes=[:Q]
  qkv_projection_fusion              Δ=  0.5    nodes=[:up]

  Gain total estimé : Δ=3.5
────────────────────────────────────────────────────────

=== Compilation ===
╔══════════════════════════════════════╗
║     NeuroDSL — CompiledPlan          ║
╚══════════════════════════════════════╝
  Namespace   : :main

In [19]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

if !haskey(NeuroDSL.CUSTOM_OPS, :flash_attn)
    NeuroDSL.register_op!(:flash_attn, NeuroDSL._dispatch_flash_attn!)
end

d_model, d_ff = 512, 2048
seq_len = 128

Random.seed!(42)
x = CUDA.rand(Float32, seq_len, d_model)

W_q = CUDA.rand(Float32, d_model, d_model)
W_k = CUDA.rand(Float32, d_model, d_model)
W_v = CUDA.rand(Float32, d_model, d_model)
W_o = CUDA.rand(Float32, d_model, d_model)

W_up   = CUDA.rand(Float32, d_model, d_ff)
W_gate = CUDA.rand(Float32, d_model, d_ff)
W_down = CUDA.rand(Float32, d_ff, d_model)

function build_neuro_block(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, x; namespace=name)
    set!(g, :W_q, W_q; namespace=name)
    set!(g, :W_k, W_k; namespace=name)
    set!(g, :W_v, W_v; namespace=name)
    set!(g, :W_o, W_o; namespace=name)
    set!(g, :W_up, W_up; namespace=name)
    set!(g, :W_gate, W_gate; namespace=name)
    set!(g, :W_down, W_down; namespace=name)

    addrule!(g, GraphRule(:Q, [:x, :W_q], :matmul; namespace=name))
    addrule!(g, GraphRule(:K, [:x, :W_k], :matmul; namespace=name))
    addrule!(g, GraphRule(:V, [:x, :W_v], :matmul; namespace=name))
    addrule!(g, GraphRule(:attn, [:Q, :K, :V], :flash_attn;
                         attrs=Dict(:d_head=>d_model, :causal=>true), namespace=name))
    addrule!(g, GraphRule(:proj, [:attn, :W_o], :matmul; namespace=name))
    addrule!(g, GraphRule(:res1, [:proj, :x], :add; namespace=name))
    addrule!(g, GraphRule(:gate, [:res1, :W_gate], :matmul; namespace=name))
    addrule!(g, GraphRule(:up,   [:res1, :W_up],   :matmul; namespace=name))
    addrule!(g, GraphRule(:ffn,  [:gate, :up],  :swiglu; namespace=name))
    addrule!(g, GraphRule(:down, [:ffn, :W_down], :matmul; namespace=name))
    addrule!(g, GraphRule(:res2, [:down, :res1], :add; namespace=name))
    return g
end

g_neuro = build_neuro_block(:main)

function sigmoid_fast(x)
    return 1f0 ./ (1f0 .+ exp.(-x))
end

function flux_llama_block(x_flux)
    Q = x_flux * W_q
    K = x_flux * W_k
    V = x_flux * W_v
    attn = flash_attn_fwd!(Backend.CUDADevice(), similar(Q), Q, K, V, d_model; causal=true) |> first
    proj = attn * W_o
    res1 = proj + x_flux
    gate = res1 * W_gate
    up   = res1 * W_up
    ffn  = gate .* sigmoid_fast(gate) .* up
    down = ffn * W_down
    res2 = down + res1
    return res2
end

x_flux = x   # (128, 512)

# Warm-up (avec invalidation pour NeuroDSL)
for _ in 1:10
    set!(g_neuro, :x, x)   # invalidation
    demand!(g_neuro, :res2; ctx_store=CtxStore())
    flux_llama_block(x_flux)
end

# Benchmark (avec invalidation à chaque itération)
N = 50
CUDA.@sync t_neuro = @elapsed for _ in 1:N
    set!(g_neuro, :x, x)   # force le recalcul
    demand!(g_neuro, :res2; ctx_store=CtxStore())
end
CUDA.@sync t_flux = @elapsed for _ in 1:N
    flux_llama_block(x_flux)
end

t_neuro_avg = t_neuro / N * 1000
t_flux_avg  = t_flux / N * 1000

@printf("NeuroDSL (non compilé) : %.4f ms/appel\n", t_neuro_avg)
@printf("Flux.jl                : %.4f ms/appel\n", t_flux_avg)

if t_neuro < t_flux
    @printf("✅ NeuroDSL est x%.2f plus rapide que Flux\n", (t_flux/N) / (t_neuro/N))
else
    @printf("⚠️ Flux est x%.2f plus rapide que NeuroDSL\n", (t_neuro/N) / (t_flux/N))
end

out_neuro = Array(demand!(g_neuro, :res2; ctx_store=CtxStore()))
out_flux  = Array(flux_llama_block(x_flux))
diff = maximum(abs.(out_neuro .- out_flux))
println("Différence maximale : ", diff)
if diff < 1e-3
    println("✅ Sorties identiques (tolérance 1e-3)")
else
    println("⚠️ Écart significatif, vérifier les opérations")
end

NeuroDSL (non compilé) : 1.6949 ms/appel
Flux.jl                : 1.7158 ms/appel
✅ NeuroDSL est x1.01 plus rapide que Flux
Différence maximale : 0.0
✅ Sorties identiques (tolérance 1e-3)


# Aprés compilation

In [20]:
using NeuroDSL, Random, Printf, LinearAlgebra, CUDA

if !haskey(NeuroDSL.CUSTOM_OPS, :flash_attn)
    NeuroDSL.register_op!(:flash_attn, NeuroDSL._dispatch_flash_attn!)
end

d_model, d_ff = 512, 2048
seq_len = 128

# ── Init réaliste : centrée sur 0, échelle Xavier-like ──────
# (avec CUDA.rand brut, moyenne 0.5 non nulle -> les magnitudes explosent
#  de façon déterministe à travers les matmuls successifs, cf. diagnostic)
scale_dm = Float32(1 / sqrt(d_model))
scale_ff = Float32(1 / sqrt(d_ff))

Random.seed!(42)
x = (CUDA.rand(Float32, seq_len, d_model) .- 0.5f0) .* scale_dm

W_q = (CUDA.rand(Float32, d_model, d_model) .- 0.5f0) .* scale_dm
W_k = (CUDA.rand(Float32, d_model, d_model) .- 0.5f0) .* scale_dm
W_v = (CUDA.rand(Float32, d_model, d_model) .- 0.5f0) .* scale_dm
W_o = (CUDA.rand(Float32, d_model, d_model) .- 0.5f0) .* scale_dm
W_up   = (CUDA.rand(Float32, d_model, d_ff) .- 0.5f0) .* scale_dm
W_gate = (CUDA.rand(Float32, d_model, d_ff) .- 0.5f0) .* scale_dm
W_down = (CUDA.rand(Float32, d_ff, d_model) .- 0.5f0) .* scale_ff

# ═══════════════════════════════════════════════════════════
# 1. NeuroDSL avec QKV fusionné manuellement
# ═══════════════════════════════════════════════════════════
W_qkv_combined = hcat(W_q, W_k, W_v)   # (512, 1536)

@defop fused_qkv_projection (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    A, B = inputs[1], inputs[2]   # x, W_qkv
    LinearAlgebra.mul!(out, A, B)   # (128, 1536)
end

# Règles locales et explicites : pas de réécriture automatique cachée
# (aucun besoin de rewrite rules ici, la fusion QKV est déjà faite à la main)
compiler_rules = RewriteRule[]

function build_neuro_fused_block(name)
    g = NeuroGraph(device=Backend.CUDADevice())
    activate!(g, name)
    set!(g, :x, x; namespace=name)
    set!(g, :W_qkv, W_qkv_combined; namespace=name)
    set!(g, :W_o, W_o; namespace=name)
    set!(g, :W_up, W_up; namespace=name)
    set!(g, :W_gate, W_gate; namespace=name)
    set!(g, :W_down, W_down; namespace=name)

    addrule!(g, GraphRule(:qkv, [:x, :W_qkv], :fused_qkv_projection; namespace=name))
    addrule!(g, GraphRule(:Q, [:qkv], :slice_cols;
                         attrs=Dict(:start_col=>1, :end_col=>d_model), namespace=name))
    addrule!(g, GraphRule(:K, [:qkv], :slice_cols;
                         attrs=Dict(:start_col=>d_model+1, :end_col=>2*d_model), namespace=name))
    addrule!(g, GraphRule(:V, [:qkv], :slice_cols;
                         attrs=Dict(:start_col=>2*d_model+1, :end_col=>3*d_model), namespace=name))

    addrule!(g, GraphRule(:attn, [:Q, :K, :V], :flash_attn;
                         attrs=Dict(:d_head=>d_model, :causal=>true), namespace=name))
    addrule!(g, GraphRule(:proj, [:attn, :W_o], :matmul; namespace=name))
    addrule!(g, GraphRule(:res1, [:proj, :x], :add; namespace=name))
    addrule!(g, GraphRule(:gate, [:res1, :W_gate], :matmul; namespace=name))
    addrule!(g, GraphRule(:up,   [:res1, :W_up],   :matmul; namespace=name))
    addrule!(g, GraphRule(:ffn,  [:gate, :up],  :swiglu; namespace=name))
    addrule!(g, GraphRule(:down, [:ffn, :W_down], :matmul; namespace=name))
    addrule!(g, GraphRule(:res2, [:down, :res1], :add; namespace=name))
    return g
end

g_neuro = build_neuro_fused_block(:main)
plan = compile(g_neuro, CompilerConfig(rules=compiler_rules))

# ═══════════════════════════════════════════════════════════
# 2. Flux.jl (non fusionné, trois projections séparées)
# ═══════════════════════════════════════════════════════════
function sigmoid_fast(x)
    return 1f0 ./ (1f0 .+ exp.(-x))
end

function flux_llama_block(x_flux)
    Q = x_flux * W_q
    K = x_flux * W_k
    V = x_flux * W_v
    attn = flash_attn_fwd!(Backend.CUDADevice(), similar(Q), Q, K, V, d_model; causal=true) |> first
    proj = attn * W_o
    res1 = proj + x_flux
    gate = res1 * W_gate
    up   = res1 * W_up
    ffn  = gate .* sigmoid_fast(gate) .* up
    down = ffn * W_down
    res2 = down + res1
    return res2
end

x_flux = x

for _ in 1:10
    set!(g_neuro, :x, x)
    plan(g_neuro, :res2)
    flux_llama_block(x_flux)
end

N = 50
CUDA.@sync t_neuro = @elapsed for _ in 1:N
    set!(g_neuro, :x, x)
    plan(g_neuro, :res2)
end
CUDA.@sync t_flux = @elapsed for _ in 1:N
    flux_llama_block(x_flux)
end

t_neuro_avg = t_neuro / N * 1000
t_flux_avg  = t_flux / N * 1000

@printf("NeuroDSL (QKV fusionné) : %.4f ms/appel\n", t_neuro_avg)
@printf("Flux.jl (non fusionné)  : %.4f ms/appel\n", t_flux_avg)

if t_neuro < t_flux
    @printf("✅ NeuroDSL est x%.2f plus rapide que Flux\n", (t_flux/N) / (t_neuro/N))
else
    @printf("⚠️ Flux est x%.2f plus rapide que NeuroDSL\n", (t_neuro/N) / (t_flux/N))
end

# ── Comparaison en tolérance relative (la seule pertinente en float32) ──
out_neuro = Array(plan(g_neuro, :res2))
out_flux  = Array(flux_llama_block(x_flux))
abs_diff = maximum(abs.(out_neuro .- out_flux))
rel_diff = maximum(abs.(out_neuro .- out_flux) ./ (abs.(out_flux) .+ 1f-6))
println("Écart absolu maximal  : ", abs_diff)
println("Écart relatif maximal : ", rel_diff)
if rel_diff < 1f-3
    println("✅ Sorties identiques (tolérance relative 1e-3)")
else
    println("⚠️ Écart relatif supérieur à 1e-3 — vérifier les calculs")
end


✅ Op :fused_qkv_projection registered


[ Info: Aucune fusion applicable.


NeuroDSL (QKV fusionné) : 2.5686 ms/appel
Flux.jl (non fusionné)  : 1.6973 ms/appel
⚠️ Flux est x1.51 plus rapide que NeuroDSL
Écart absolu maximal  : 1.8626451e-9
Écart relatif maximal : 3.0182258e-5
✅ Sorties identiques (tolérance relative 1e-3)
